In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter
from datetime import datetime

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

from ai_agents.agents.email_finder import run_batch, run_single, SourceType, LeadStatus
from ai_agents.agents.email_finder.graph import build_graph
from google_utils.google_sheet import GoogleSheetService

from dotenv import load_dotenv
load_dotenv()


CHECKPOINT_FILE = "email_finder_guest_checkpoint.json"
BATCH_SIZE = 50
CONCURRENCY = 5
SKIP_SHEET = "List Info"

In [ ]:
# ── v2 Graph structure ──
# canonical_builder
#   ├─ (has existing_email, no website) → validate_existing_email
#   │     ├─ (email validated ≥0.85) → END
#   │     ├─ (has website) → discover_urls
#   │     └─ (no website) → perplexity_discovery
#   ├─ (has website) → discover_urls → crawl_page (fan-out) → resolve_best_email
#   │     ├─ (email found) → END
#   │     ├─ (no candidates + has FB link) → fb_crawler
#   │     │     ├─ (email found) → END
#   │     │     └─ → perplexity_discovery → END
#   │     └─ → perplexity_discovery → END
#   └─ (no website) → perplexity_discovery → END

graph = build_graph()
print("Graph compiled OK")
print(f"Nodes: {list(graph.get_graph().nodes)}")

try:
    print(graph.get_graph().draw_ascii())
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
def load_checkpoint(spreadsheet_id: str) -> dict:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            data = json.load(f)
            if data.get("spreadsheet_id") == spreadsheet_id:
                print(f"Resuming from checkpoint — {len(data['processed_keys'])} guests already processed")
                return data
    return {
        "spreadsheet_id": spreadsheet_id,
        "processed_keys": [],
        "found_count": 0,
        "not_found_count": 0,
        "failed_count": 0,
        "last_updated": None,
    }


def save_checkpoint(checkpoint: dict) -> None:
    checkpoint["last_updated"] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2)


def clear_checkpoint() -> None:
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared")

In [ ]:
SOCIAL_PLATFORM_MAP = {
    0: "instagram",
    1: "linkedin",
    2: "facebook",
    3: "twitter",
    4: "youtube",
}


def _extract_fb_from_other_social(other_social: str) -> str:
    """Extract facebook URL from the guest_other_social column."""
    if not other_social:
        return ""
    for url in other_social.split(","):
        url = url.strip()
        if "facebook.com" in url.lower():
            return url
    return ""


def _guest_key(name: str, company: str) -> str:
    """Dedup key for a guest — lowercase name + company."""
    return f"{name.strip().lower()}|{company.strip().lower()}"


def build_guest_raw_rows(row: dict) -> list[dict]:
    """
    Convert a sheet row into one or more raw_row dicts for the email finder.

    Handles multi-guest rows by splitting comma-separated guest_names and
    aligning with other guest_* columns by index.

    "Appeared On Podcast" provides context to help identify the guest but
    the canonical_builder prompt instructs the LLM to treat it as context
    only and never extract the podcast host's information.

    Returns empty list if no guest entity data is present.
    """
    entity_details = (row.get("guest_entity_api_details") or "").strip()
    guest_names_raw = (row.get("guest_names") or "").strip()

    if not entity_details and not guest_names_raw:
        return []

    podcast_name = row.get("podcast_name") or row.get("podcast_title") or ""

    # Try to use the JSON directly for richer data
    if entity_details:
        try:
            entity = json.loads(entity_details)
            social_links = entity.get("social_links") or []
            socials = {}
            other_social = []
            for link in social_links:
                url = (link.get("url") or "").strip()
                if not url:
                    continue
                platform_id = link.get("platform")
                platform_name = SOCIAL_PLATFORM_MAP.get(platform_id)
                if platform_name and platform_name not in socials:
                    socials[platform_name] = url
                elif not platform_name:
                    other_social.append(url)

            raw_row = {
                "Guest Name": entity.get("entity_name") or "",
                "Company": entity.get("company") or "",
                "Occupation": entity.get("occupation") or "",
                "Industry": entity.get("industry") or "",
                "Website": entity.get("url") or "",
                "Instagram": socials.get("instagram", ""),
                "LinkedIn": socials.get("linkedin", ""),
                "Facebook": socials.get("facebook", ""),
                "Twitter": socials.get("twitter", ""),
                "YouTube": socials.get("youtube", ""),
                "Other Social": ", ".join(other_social),
                "Appeared On Podcast": podcast_name,
                "_entity_id": entity.get("entity_id") or "",
                "_dedup_key": _guest_key(
                    entity.get("entity_name") or "",
                    entity.get("company") or "",
                ),
            }
            return [raw_row]
        except (json.JSONDecodeError, TypeError):
            pass  # Fall through to parsed columns

    # Fallback: use pre-parsed guest_* columns
    names = [n.strip() for n in guest_names_raw.split(",") if n.strip()]
    companies = [c.strip() for c in (row.get("guest_companies") or "").split(",")]
    occupations = [o.strip() for o in (row.get("guest_occupations") or "").split(",")]
    industries = [ind.strip() for ind in (row.get("guest_industries") or "").split(",")]

    # Social columns are typically for the first/only guest
    instagram = (row.get("guest_instagram") or "").strip()
    linkedin = (row.get("guest_linkedin") or "").strip()
    twitter = (row.get("guest_twitter") or "").strip()
    website = (row.get("guest_website") or "").strip()
    other_social = (row.get("guest_other_social") or "").strip()
    facebook = _extract_fb_from_other_social(other_social)

    raw_rows = []
    for i, name in enumerate(names):
        company = companies[i] if i < len(companies) else ""
        occupation = occupations[i] if i < len(occupations) else ""
        industry = industries[i] if i < len(industries) else ""

        raw_row = {
            "Guest Name": name,
            "Company": company,
            "Occupation": occupation,
            "Industry": industry,
            "Website": website if i == 0 else "",
            "Instagram": instagram if i == 0 else "",
            "LinkedIn": linkedin if i == 0 else "",
            "Facebook": facebook if i == 0 else "",
            "Twitter": twitter if i == 0 else "",
            "Other Social": other_social if i == 0 else "",
            "Appeared On Podcast": podcast_name,
            "_entity_id": "",
            "_dedup_key": _guest_key(name, company),
        }
        raw_rows.append(raw_row)

    return raw_rows


def collect_guest_rows(
    sheet_service: GoogleSheetService,
    spreadsheet_id: str,
    already_processed: set[str],
) -> list[dict]:
    """
    Read all sheets (except List Info), extract guest rows, deduplicate.

    Returns list of raw_row dicts ready for run_batch.
    """
    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    assert success, f"Cannot list sheets: {sheet_names}"

    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    print(f"Sheets to process: {sheets_to_process}")

    seen_keys: set[str] = set(already_processed)
    all_raw_rows: list[dict] = []
    stats = {"total_rows": 0, "no_guest_data": 0, "duplicates": 0}

    for sheet_name in sheets_to_process:
        success, df = sheet_service.get_sheet_data(spreadsheet_id, sheet_name)
        if not success:
            print(f"  [{sheet_name}] Error: {df}")
            continue

        sheet_count = 0
        for _, row in df.iterrows():
            stats["total_rows"] += 1
            guest_rows = build_guest_raw_rows(row.to_dict())

            if not guest_rows:
                stats["no_guest_data"] += 1
                continue

            for raw_row in guest_rows:
                key = raw_row["_dedup_key"]
                if key in seen_keys:
                    stats["duplicates"] += 1
                    continue
                seen_keys.add(key)
                raw_row["_source_sheet"] = sheet_name
                all_raw_rows.append(raw_row)
                sheet_count += 1

        print(f"  [{sheet_name}] {sheet_count} unique guests extracted")

    print(f"\nCollection summary:")
    print(f"  Total episode rows scanned: {stats['total_rows']}")
    print(f"  Rows without guest data:    {stats['no_guest_data']}")
    print(f"  Duplicate guests skipped:   {stats['duplicates']}")
    print(f"  Unique guests to process:   {len(all_raw_rows)}")

    return all_raw_rows

In [ ]:
def _format_email_source(best_email: dict) -> str:
    source = (best_email.get("source") or "").strip()
    note = (best_email.get("note") or "").strip()
    if source and note:
        return f"{source} — {note}"
    if note:
        return note
    return source


def result_to_sheet_row(raw_row: dict, result: dict) -> list:
    best_email = result.get("best_email") or {}
    if isinstance(best_email, dict):
        email = best_email.get("email", "")
        source_display = _format_email_source(best_email)
        confidence = best_email.get("confidence", "")
    else:
        email = ""
        source_display = ""
        confidence = ""

    nodes = result.get("nodes_executed") or []
    errors = result.get("errors") or []

    return [
        raw_row.get("Guest Name", ""),
        raw_row.get("Company", ""),
        raw_row.get("Occupation", ""),
        raw_row.get("Industry", ""),
        raw_row.get("Website", ""),
        raw_row.get("Appeared On Podcast", ""),
        email,
        source_display,
        str(confidence),
        result.get("status", ""),
        " → ".join(nodes),
        "; ".join(errors) if errors else "",
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    ]

In [ ]:
def _create_sheet_tab(sheet_service: GoogleSheetService, spreadsheet_id: str, sheet_name: str) -> None:
    """Create a new sheet tab if it doesn't exist."""
    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    if success and sheet_name in sheet_names:
        return
    sheet_service.service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},
    ).execute()
    print(f"Created sheet tab '{sheet_name}'")


def _ensure_email_finder_headers(
    sheet_service: GoogleSheetService,
    spreadsheet_id: str,
    sheet_name: str,
) -> None:
    """Create the Email_Finder sheet tab (if needed) and add headers if empty."""
    _create_sheet_tab(sheet_service, spreadsheet_id, sheet_name)
    success, values = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A1:M1")
    if not success or not values:
        headers = [[
            "Guest Name",
            "Company",
            "Occupation",
            "Industry",
            "Website",
            "Podcast Name",
            "Email Found",
            "Email Source",
            "Confidence",
            "Status",
            "Nodes Executed",
            "Errors",
            "Processed At",
        ]]
        sheet_service.append_rows(spreadsheet_id, sheet_name, headers)
        print(f"Added headers to {sheet_name}")


def run_email_finder_guest_pipeline(
    spreadsheet_id: str,
    email_finder_sheet: str = "Email_Finder",
    batch_size: int = BATCH_SIZE,
    concurrency: int = CONCURRENCY,
    dry_run: bool = False,
) -> dict:
    """
    Podscan Guest email finder pipeline (v2).

    Reads all sheets (except List Info), extracts unique guests from rows
    that have guest_entity_api_details, deduplicates by name+company,
    and runs the email finder graph for each guest.

    Results are written to the Email_Finder sheet in the same spreadsheet.
    """
    sheet_service = GoogleSheetService()
    checkpoint = load_checkpoint(spreadsheet_id)
    already_processed = set(checkpoint["processed_keys"])

    # ── Collect all unique guests across sheets ──
    print("Scanning sheets for guest data...")
    all_raw_rows = collect_guest_rows(sheet_service, spreadsheet_id, already_processed)

    if not all_raw_rows:
        print("Nothing to process — all guests already handled or no guest data found")
        return checkpoint

    # Ensure Email_Finder sheet tab exists and has headers
    if not dry_run:
        _ensure_email_finder_headers(sheet_service, spreadsheet_id, email_finder_sheet)

    # ── Per-node stats ──
    node_stats: Counter = Counter()
    source_stats: Counter = Counter()

    # ── Process in batches ──
    for batch_start in range(0, len(all_raw_rows), batch_size):
        batch_rows = all_raw_rows[batch_start: batch_start + batch_size]
        batch_num = batch_start // batch_size + 1
        total_batches = (len(all_raw_rows) + batch_size - 1) // batch_size

        print(f"\nBatch {batch_num}/{total_batches} — {len(batch_rows)} guests")

        results = run_batch(
            rows=batch_rows,
            source_type=SourceType.PODSCAN_GUEST,
            concurrency=concurrency,
        )

        found_rows = []
        found_keys = []

        for raw_row, result in zip(batch_rows, results):
            status = result.get("status", "")
            dedup_key = raw_row["_dedup_key"]

            # Track node execution
            for node in (result.get("nodes_executed") or []):
                node_stats[node] += 1

            # Track email source
            best = result.get("best_email") or {}
            if isinstance(best, dict) and best.get("source"):
                source_stats[best["source"].split(" \u2014 ")[0]] += 1

            if status == "email_found":
                found_rows.append(result_to_sheet_row(raw_row, result))
                found_keys.append(dedup_key)
                checkpoint["found_count"] += 1
            else:
                checkpoint["processed_keys"].append(dedup_key)
                if status == "failed":
                    checkpoint["failed_count"] += 1
                else:
                    checkpoint["not_found_count"] += 1

        # Write found emails to Email_Finder sheet
        if found_rows:
            if not dry_run:
                success, msg = sheet_service.append_rows(
                    spreadsheet_id, email_finder_sheet, found_rows
                )
                print(f"  \u2192 Wrote {len(found_rows)} emails to {email_finder_sheet}: {msg}")
                if success:
                    checkpoint["processed_keys"].extend(found_keys)
                else:
                    checkpoint["found_count"] -= len(found_rows)
                    print(f"  \u26a0 Sheet write failed — {len(found_rows)} found emails will be retried next run")
            else:
                checkpoint["processed_keys"].extend(found_keys)

        save_checkpoint(checkpoint)
        print(f"  \u2192 Checkpoint saved | Found: {checkpoint['found_count']} | Not found: {checkpoint['not_found_count']} | Failed: {checkpoint['failed_count']}")

    # ── Final summary ──
    print(f"\n{'='*50}")
    print(f"Pipeline complete")
    print(f"{'='*50}")
    print(f"  Found:     {checkpoint['found_count']}")
    print(f"  Not found: {checkpoint['not_found_count']}")
    print(f"  Failed:    {checkpoint['failed_count']}")

    if node_stats:
        print(f"\n  Node execution counts:")
        for node, count in node_stats.most_common():
            print(f"    {node}: {count}")

    if source_stats:
        print(f"\n  Email sources:")
        for source, count in source_stats.most_common():
            print(f"    {source}: {count}")

    if not dry_run:
        clear_checkpoint()

    return checkpoint

In [ ]:
# ── Quick single-guest test ──
# Uncomment to test the graph with a single guest before running the full pipeline.

# test_row = {
#     "Guest Name": "Jeremy Dyer",
#     "Company": "Starting Point Capital",
#     "Occupation": "Investor and Fund Manager",
#     "Industry": "Real Estate Investment",
#     "Website": "",
#     "Instagram": "https://www.instagram.com/startingpointcapital/",
#     "LinkedIn": "https://www.linkedin.com/in/jeremydyer",
#     "Facebook": "https://www.facebook.com/startingpointcapital",
# }
# result = run_single(test_row, SourceType.PODSCAN_GUEST)
# print(f"Status: {result.get('status')}")
# print(f"Nodes:  {' \u2192 '.join(result.get('nodes_executed', []))}")
# print(f"Email:  {result.get('best_email', {})}")
# print(f"Errors: {result.get('errors', [])}")

In [ ]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1pPF2clctk6NxAfI5AjgUJhasy8YrT1b_9XEf6MGbJe0/edit?usp=sharing"  # Paste the podscan guest spreadsheet URL here
sheet_service = GoogleSheetService()
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

# Preview: list sheets and sample guest data before running
success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
if success:
    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    print(f"All sheets: {sheet_names}")
    print(f"Will process: {sheets_to_process}")

    # Sample first sheet to verify column structure
    if sheets_to_process:
        ok, df = sheet_service.get_sheet_data(spreadsheet_id, sheets_to_process[0])
        if ok:
            print(f"\nColumns in '{sheets_to_process[0]}': {list(df.columns)}")
            guest_rows = df[df["guest_names"].notna() & (df["guest_names"].str.strip() != "")]
            print(f"Rows with guest data: {len(guest_rows)} / {len(df)}")

In [ ]:
result = run_email_finder_guest_pipeline(
    spreadsheet_id=spreadsheet_id,
    dry_run=False,
    batch_size=50,
    concurrency=5,
)